In [ ]:
import os 
import sys 
import torch 
import torch .nn as nn 
import torch .optim as optim 
import numpy as np 
from pathlib import Path 


project_root =Path .cwd ()
while not (project_root /'requirements.txt').exists ():
    if project_root ==project_root .parent :
        raise RuntimeError ("Could not find project root (looked for requirements.txt)")
    project_root =project_root .parent 
sys .path .insert (0 ,str (project_root ))


from src .models .main .classical import ShallowCNN ,ClassicalModel 
from src .models .main .models import QuantumModel ,ExtentedQuantumModel 
from src .models .fusion import OnlineFusionModel 


from src .utils .dataloader import BreastMNISTLoader ,INbreastLoader ,BUSUCLMLoader 


from src .utils .training import train_epoch ,validate_model ,test_model ,training_loop 


device =torch .device ('cuda'if torch .cuda .is_available ()else 'cpu')
print (f"Using device: {device }")
print (f"Project root: {project_root }")

In [ ]:
DATASET ='BUSUCLM'

CLASSICAL_MODEL_TYPE ='shallow'
QUANTUM_MODEL_TYPE ='extended_quantum'

CLASSICAL_PRETRAINED_PATH =str (project_root /'experiments'/'models'/'classical'/'BUSUCLM'/'resnet'/'best_resnet.pth')
QUANTUM_PRETRAINED_PATH =str (project_root /'experiments'/'models'/'quantum'/'trainable'/'BUSUCLM'/'checkpoints'/'model_epoch5.pth')


FREEZE_CLASSICAL =False 
FREEZE_QUANTUM =False 


BATCH_SIZE =32 
IMAGE_SIZE =28 
NUM_EPOCHS =10 
FEATURE_DIM =128 
NUM_CLASSES =2 

LR_CLASSICAL =1e-4 
LR_QUANTUM =1e-2 
LR_FUSION =1e-3 

N_QUBITS =4 
N_LAYERS =1 
PATCH_SIZE =2 

OUTPUT_DIR =str (project_root /'experiments'/'models'/'fusion'/'online'/DATASET /f'{CLASSICAL_MODEL_TYPE }_{QUANTUM_MODEL_TYPE }')
os .makedirs (OUTPUT_DIR ,exist_ok =True )

print (f"\n{'='*60 }")
print (f"Configuração:")
print (f"  Dataset: {DATASET }")
print (f"  Classical Model: {CLASSICAL_MODEL_TYPE }")
print (f"  Quantum Model: {QUANTUM_MODEL_TYPE }")
print (f"  Feature Dimension: {FEATURE_DIM }")
print (f"  Learning Rates: Classical={LR_CLASSICAL }, Quantum={LR_QUANTUM }, Fusion={LR_FUSION }")
print (f"  Output Directory: {OUTPUT_DIR }")
print (f"{'='*60 }\n")

In [ ]:

if DATASET =='BreastMNIST':
    loader =BreastMNISTLoader (batch_size =BATCH_SIZE ,image_size =IMAGE_SIZE )
elif DATASET =='INbreast':
    loader =INbreastLoader (batch_size =BATCH_SIZE ,image_size =IMAGE_SIZE ,
    root_dir =str (project_root /'src'/'data'/'INbreast'/'png'))
elif DATASET =='BUSUCLM':
    loader =BUSUCLMLoader (batch_size =BATCH_SIZE ,image_size =IMAGE_SIZE ,
    root_dir =str (project_root /'src'/'data'/'BUS-UCLM'/'png'))
else :
    raise ValueError (f"Dataset não suportado: {DATASET }")


train_loader ,test_loader ,val_loader =loader .get_dataloaders ()

print (f"✓ Dataset {DATASET } carregado:")
print (f"  Train batches: {len (train_loader )}")
print (f"  Val batches: {len (val_loader )}")
print (f"  Test batches: {len (test_loader )}")

In [ ]:

if CLASSICAL_MODEL_TYPE =='shallow':
    classical_model =ShallowCNN (num_classes =NUM_CLASSES ,input_channels =1 )
elif CLASSICAL_MODEL_TYPE =='resnet':
    classical_model =ClassicalModel (num_classes =NUM_CLASSES )
else :
    raise ValueError (f"Tipo de modelo clássico não suportado: {CLASSICAL_MODEL_TYPE }")

print (f"✓ Modelo clássico criado: {CLASSICAL_MODEL_TYPE }")


if QUANTUM_MODEL_TYPE =='quantum':

    rand_params =np .random .uniform (high =2 *np .pi ,size =(N_QUBITS ,N_LAYERS ))
    quantum_model =QuantumModel (
    n_qubits =N_QUBITS ,
    n_layers =N_LAYERS ,
    rand_params =rand_params ,
    output_size =(14 ,14 ),
    patch_size =PATCH_SIZE ,
    num_classes =NUM_CLASSES 
    )
elif QUANTUM_MODEL_TYPE =='extended_quantum':

    quantum_model =ExtentedQuantumModel (
    n_qubits =N_QUBITS ,
    n_layers =N_LAYERS ,
    num_classes =NUM_CLASSES ,
    patch_size =PATCH_SIZE ,
    mode ="2d",
    stride =PATCH_SIZE ,
    use_parallel =True 
    )
else :
    raise ValueError (f"Tipo de modelo quântico não suportado: {QUANTUM_MODEL_TYPE }")

print (f"✓ Modelo quântico criado: {QUANTUM_MODEL_TYPE }")


fusion_model =OnlineFusionModel (
classical_model =classical_model ,
quantum_model =quantum_model ,
feature_dim =FEATURE_DIM ,
num_classes =NUM_CLASSES ,
freeze_classical =FREEZE_CLASSICAL ,
freeze_quantum =FREEZE_QUANTUM 
)


if CLASSICAL_PRETRAINED_PATH or QUANTUM_PRETRAINED_PATH :
    fusion_model .load_pretrained_weights (
    classical_path =CLASSICAL_PRETRAINED_PATH ,
    quantum_path =QUANTUM_PRETRAINED_PATH ,
    device =device 
    )

fusion_model =fusion_model .to (device )
print (f"\n✓ Modelo de fusão criado e movido para {device }")


total_params =sum (p .numel ()for p in fusion_model .parameters ())
trainable_params =sum (p .numel ()for p in fusion_model .parameters ()if p .requires_grad )
print (f"  Total de parâmetros: {total_params :,}")
print (f"  Parâmetros treináveis: {trainable_params :,}")

In [ ]:

print ("Inicializando projeções...")
fusion_model .eval ()
with torch .no_grad ():
    dummy_input =torch .randn (1 ,1 ,IMAGE_SIZE ,IMAGE_SIZE ).to (device )
    _ =fusion_model (dummy_input )
fusion_model .train ()
print ("✓ Projeções inicializadas\n")


param_groups =fusion_model .get_parameter_groups (
lr_classical =LR_CLASSICAL ,
lr_quantum =LR_QUANTUM ,
lr_fusion =LR_FUSION 
)

optimizer =optim .Adam (param_groups )


criterion =nn .CrossEntropyLoss ()

print (f"✓ Otimizador configurado com {len (param_groups )} grupos de parâmetros")
print (f"✓ Loss function: CrossEntropyLoss\n")


group_names =['Clássico','Quântico','Fusão']
for i ,group in enumerate (param_groups ):
    num_params =sum (p .numel ()for p in group ['params'])
    num_tensors =len (group ['params'])
    name =group_names [i ]if i <len (group_names )else f"Grupo {i +1 }"
    print (f"  {name }:")
    print (f"    Learning Rate: {group ['lr']}")
    print (f"    Parâmetros Treináveis: {num_params :,}")
    print (f"    Número de Tensores: {num_tensors }")


print (f"\n{'='*60 }")
print ("DEBUG: Verificação de Parâmetros por Modelo")
print (f"{'='*60 }")

classical_params =sum (p .numel ()for p in fusion_model .classical_model .parameters ()if p .requires_grad )
quantum_params =sum (p .numel ()for p in fusion_model .quantum_model .parameters ()if p .requires_grad )
projection_params =0 
if fusion_model .classical_projection is not None :
    projection_params +=sum (p .numel ()for p in fusion_model .classical_projection .parameters ())
if fusion_model .quantum_projection is not None :
    projection_params +=sum (p .numel ()for p in fusion_model .quantum_projection .parameters ())
fusion_layers_params =sum (p .numel ()for p in fusion_model .fusion_norm .parameters ())
fusion_layers_params +=sum (p .numel ()for p in fusion_model .fusion_fc .parameters ())

print (f"Modelo Clássico: {classical_params :,} parâmetros")
print (f"Modelo Quântico: {quantum_params :,} parâmetros")
print (f"Projeções (classical + quantum): {projection_params :,} parâmetros")
print (f"Camadas de Fusão (norm + fc): {fusion_layers_params :,} parâmetros")
print (f"TOTAL: {classical_params +quantum_params +projection_params +fusion_layers_params :,} parâmetros")
print (f"{'='*60 }\n")

In [ ]:

history =training_loop (
model =fusion_model ,
train_loader =train_loader ,
val_loader =val_loader ,
criterion =criterion ,
optimizer =optimizer ,
device =device ,
num_epochs =NUM_EPOCHS ,
save_dir =OUTPUT_DIR ,
model_name =f"fusion_{CLASSICAL_MODEL_TYPE }_{QUANTUM_MODEL_TYPE }",
extract_features =False ,
save_best_only =True ,
save_all_epochs =False 
)

print ("\n"+"="*60 )
print ("✓ Treinamento concluído!")
print ("="*60 )

In [ ]:

best_model_path =os .path .join (OUTPUT_DIR ,f"best_fusion_{CLASSICAL_MODEL_TYPE }_{QUANTUM_MODEL_TYPE }.pth")

if os .path .exists (best_model_path ):
    fusion_model .load_state_dict (torch .load (best_model_path ,map_location =device ))
    print (f"✓ Melhor modelo carregado de {best_model_path }")
else :
    print (f"⚠ Arquivo de modelo não encontrado. Usando modelo atual.")


hyperparams ={
'dataset':DATASET ,
'classical_model':CLASSICAL_MODEL_TYPE ,
'quantum_model':QUANTUM_MODEL_TYPE ,
'feature_dim':FEATURE_DIM ,
'batch_size':BATCH_SIZE ,
'lr_classical':LR_CLASSICAL ,
'lr_quantum':LR_QUANTUM ,
'lr_fusion':LR_FUSION ,
'n_qubits':N_QUBITS ,
'n_layers':N_LAYERS ,
'patch_size':PATCH_SIZE 
}

output_file =os .path .join (OUTPUT_DIR ,'test_results.txt')

test_loss ,test_acc ,test_prec ,test_rec ,test_f1 ,test_auc =test_model (
model =fusion_model ,
dataloader =test_loader ,
criterion =criterion ,
device =device ,
hyperparams =hyperparams ,
output_file =output_file ,
extract_features =False ,
compute_detailed_metrics =True 
)

print ("\n"+"="*60 )
print ("RESULTADOS FINAIS NO CONJUNTO DE TESTE")
print ("="*60 )
print (f"Loss: {test_loss :.4f}")
print (f"Accuracy: {test_acc :.4f}")
print (f"Precision: {test_prec :.4f}")
print (f"Recall: {test_rec :.4f}")
print (f"F1 Score: {test_f1 :.4f}")
print (f"AUC: {test_auc :.4f}")
print ("="*60 )
print (f"\nResultados detalhados salvos em: {output_file }")

In [ ]:
import umap 

fusion_model .eval ()
classical_features =[]
quantum_features =[]
fusion_features =[]
sample_labels =[]

with torch .no_grad ():
    for images ,labels in test_loader :
        images =images .to (device )

        classical_output =fusion_model .classical_model .extract_features (images )
        classical_features .append (classical_output .cpu ().numpy ())

        quantum_output =fusion_model .quantum_model .extract_features (images )
        quantum_features .append (quantum_output .cpu ().numpy ())

        fusion_output =fusion_model .extract_features (images )
        fusion_features .append (fusion_output .cpu ().numpy ())

        sample_labels .append (labels .numpy ())

classical_features =np .concatenate (classical_features ,axis =0 )
quantum_features =np .concatenate (quantum_features ,axis =0 )
fusion_features =np .concatenate (fusion_features ,axis =0 )
sample_labels =np .concatenate (sample_labels ,axis =0 ).squeeze ()

print (f"Classical features: {classical_features .shape }")
print (f"Quantum features: {quantum_features .shape }")
print (f"Fusion features: {fusion_features .shape }")
print (f"Labels: {sample_labels .shape }")

reducer =umap .UMAP (n_components =2 ,random_state =42 ,n_neighbors =15 ,min_dist =0.1 )

classical_2d =reducer .fit_transform (classical_features )
plt .figure (figsize =(10 ,8 ))
scatter =plt .scatter (classical_2d [:,0 ],classical_2d [:,1 ],
c =sample_labels ,cmap ='coolwarm',alpha =0.6 ,s =50 )
plt .colorbar (scatter ,label ='Class')
plt .xlabel ('UMAP Component 1')
plt .ylabel ('UMAP Component 2')
plt .title ('UMAP - Classical Features')
plt .grid (True ,alpha =0.3 )
plt .tight_layout ()
plt .savefig (os .path .join (OUTPUT_DIR ,'umap_classical_features.png'),dpi =300 ,bbox_inches ='tight')
plt .show ()
print (f"✓ Classical UMAP saved at {OUTPUT_DIR }/umap_classical_features.png")

quantum_2d =reducer .fit_transform (quantum_features )
plt .figure (figsize =(10 ,8 ))
scatter =plt .scatter (quantum_2d [:,0 ],quantum_2d [:,1 ],
c =sample_labels ,cmap ='coolwarm',alpha =0.6 ,s =50 )
plt .colorbar (scatter ,label ='Class')
plt .xlabel ('UMAP Component 1')
plt .ylabel ('UMAP Component 2')
plt .title ('UMAP - Quantum Features')
plt .grid (True ,alpha =0.3 )
plt .tight_layout ()
plt .savefig (os .path .join (OUTPUT_DIR ,'umap_quantum_features.png'),dpi =300 ,bbox_inches ='tight')
plt .show ()
print (f"✓ Quantum UMAP saved at {OUTPUT_DIR }/umap_quantum_features.png")

fusion_2d =reducer .fit_transform (fusion_features )
plt .figure (figsize =(10 ,8 ))
scatter =plt .scatter (fusion_2d [:,0 ],fusion_2d [:,1 ],
c =sample_labels ,cmap ='coolwarm',alpha =0.6 ,s =50 )
plt .colorbar (scatter ,label ='Class')
plt .xlabel ('UMAP Component 1')
plt .ylabel ('UMAP Component 2')
plt .title ('UMAP - Fused Features')
plt .grid (True ,alpha =0.3 )
plt .tight_layout ()
plt .savefig (os .path .join (OUTPUT_DIR ,'umap_fused_features.png'),dpi =300 ,bbox_inches ='tight')
plt .show ()
print (f"✓ Fusion UMAP saved at {OUTPUT_DIR }/umap_fused_features.png")

In [ ]:
import json 


config ={
'dataset':DATASET ,
'classical_model':CLASSICAL_MODEL_TYPE ,
'quantum_model':QUANTUM_MODEL_TYPE ,
'feature_dim':FEATURE_DIM ,
'num_classes':NUM_CLASSES ,
'batch_size':BATCH_SIZE ,
'image_size':IMAGE_SIZE ,
'num_epochs':NUM_EPOCHS ,
'lr_classical':LR_CLASSICAL ,
'lr_quantum':LR_QUANTUM ,
'lr_fusion':LR_FUSION ,
'n_qubits':N_QUBITS ,
'n_layers':N_LAYERS ,
'patch_size':PATCH_SIZE ,
'freeze_classical':FREEZE_CLASSICAL ,
'freeze_quantum':FREEZE_QUANTUM ,
'total_params':total_params ,
'trainable_params':trainable_params ,
'test_results':{
'loss':float (test_loss ),
'accuracy':float (test_acc ),
'precision':float (test_prec ),
'recall':float (test_rec ),
'f1':float (test_f1 ),
'auc':float (test_auc )
}
}

config_path =os .path .join (OUTPUT_DIR ,'config.json')
with open (config_path ,'w')as f :
    json .dump (config ,f ,indent =4 )

print (f"✓ Configuração salva em {config_path }")
print ("\n"+"="*60 )
print ("Pipeline de Fusão Online Concluída!")
print ("="*60 )